In [ ]:
# ============================================================================
# CELL 1: INSTALL DEPENDENCIES
# ============================================================================

!pip install --quiet scipy scikit-learn pandas numpy matplotlib seaborn yfinance

Please upload 'nse_indexes.csv' and 'stocks_df.csv'


Saving nse_indexes.csv to nse_indexes.csv
Saving stocks_df.csv to stocks_df.csv
Files loaded successfully
Indexes shape: (87536, 8)
Stocks shape: (4094387, 8)


In [ ]:
# ============================================================================
# CELL 2: DATA UPLOAD AND PREPROCESSING
# ============================================================================

import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# -----------------------------------------------------------------------------
# Option 1: Google Colab Upload
# -----------------------------------------------------------------------------
try:
    from google.colab import files
    import io

    print("="*60)
    print("Please upload your data files:")
    print("  1. nse_indexes.csv (Index OHLC data)")
    print("  2. stocks_df.csv (Stock OHLC data)")
    print("="*60)

    uploaded = files.upload()

    # Auto-detect file names
    def find_file(keywords):
        for fname in uploaded.keys():
            if any(k.lower() in fname.lower() for k in keywords):
                return fname
        return None

    index_file = find_file(["index", "nse"])
    stock_file = find_file(["stock"])

    if index_file is None or stock_file is None:
        print("\nAvailable files:", list(uploaded.keys()))
        raise FileNotFoundError("Could not auto-detect files. Please check filenames.")

    indexes_raw = pd.read_csv(io.BytesIO(uploaded[index_file]))
    stocks_raw = pd.read_csv(io.BytesIO(uploaded[stock_file]))

    print(f"\n✓ Loaded {index_file}: {indexes_raw.shape}")
    print(f"✓ Loaded {stock_file}: {stocks_raw.shape}")

    COLAB_ENV = True

except ImportError:
    # -----------------------------------------------------------------------------
    # Option 2: Local Environment - specify paths
    # -----------------------------------------------------------------------------
    print("Running in local environment. Loading from file paths...")

    # UPDATE THESE PATHS for local usage
    INDEX_FILE_PATH = "nse_indexes.csv"
    STOCK_FILE_PATH = "stocks_df.csv"

    indexes_raw = pd.read_csv(INDEX_FILE_PATH)
    stocks_raw = pd.read_csv(STOCK_FILE_PATH)

    print(f"\n✓ Loaded indexes: {indexes_raw.shape}")
    print(f"✓ Loaded stocks: {stocks_raw.shape}")

    COLAB_ENV = False

print("\n" + "="*60)
print("DATA LOADED SUCCESSFULLY")
print("="*60)



Fetched sector_map size from Wikipedia: 0
After yfinance fallback, mapping size: 545
Universe size after NIFTY500 strictness: 545
Using 26 tickers for optimization (after data sufficiency filter).
Stage 1: running risk-parity warm-start.
Stage 2: projecting to constraints (sector caps, liquidity, beta band, turnover)
Stage2 projection did NOT converge cleanly: Positive directional derivative for linesearch
Stage2 retry failed: Positive directional derivative for linesearch

--- Absolute-Risk (Risk-Parity) Portfolio Summary ---
Elapsed time: 105.7s
Positions: 26
Total weight: 0.9999999999999998
Top  26 :


,Stock,Weight,Sector
0,BRITANNIA,0.0385,Consumer Defensive
1,BBOX,0.0385,Technology
2,BPCL,0.0385,Energy
3,ABAN,0.0385,Energy
4,BEL,0.0385,Industrials
5,BHEL,0.0385,Industrials
6,ADANIENT,0.0385,Energy
7,GRAPHITE,0.0385,Industrials
8,FEDERALBNK,0.0385,Financial Services
9,CANFINHOME,0.0385,Financial Services



Sector exposures:


,Weight
Sector,
Basic Materials,0.2308
Financial Services,0.1923
Consumer Cyclical,0.1923
Energy,0.1154
Industrials,0.1154
Utilities,0.0769
Consumer Defensive,0.0385
Technology,0.0385



Expected annual return (approx): 0.2951 (29.51%)
Annual volatility (approx): 0.1966 (19.66%)
Portfolio beta: 0.3867
Tracking error vs benchmark (empirical): 0.3026 (30.26%)
Per-rebalance turnover (L1): 0.0000 (cap: 0.2083)

Saved: absolute_risk_parity_portfolio.csv, absolute_risk_parity_top10.csv

No sector cap violations.

No stocks exceed max weight.


Done.


In [ ]:

# ============================================================================
# CELL 3: DATA CLEANING AND ALIGNMENT
# ============================================================================

print("Cleaning and aligning data...\n")

# -----------------------------------------------------------------------------
# Clean column names
# -----------------------------------------------------------------------------
indexes_raw.columns = indexes_raw.columns.str.strip()
stocks_raw.columns = stocks_raw.columns.str.strip()

print("Index columns:", list(indexes_raw.columns))
print("Stock columns:", list(stocks_raw.columns))

# -----------------------------------------------------------------------------
# Standardize column names for stocks
# -----------------------------------------------------------------------------
stock_col_mapping = {
    'Ticker': 'Stock',
    'symbol': 'Stock',
    'Symbol': 'Stock',
    'SYMBOL': 'Stock',
    'Change Pct': 'Change_Pct',
    'ChangePct': 'Change_Pct',
    'change_pct': 'Change_Pct'
}
stocks_raw.rename(columns=stock_col_mapping, inplace=True)

# -----------------------------------------------------------------------------
# Parse dates
# -----------------------------------------------------------------------------
indexes_raw['Date'] = pd.to_datetime(indexes_raw['Date'])
stocks_raw['Date'] = pd.to_datetime(stocks_raw['Date'])

# -----------------------------------------------------------------------------
# Clean stock ticker names
# -----------------------------------------------------------------------------
stocks_raw['Stock'] = stocks_raw['Stock'].astype(str).str.upper().str.strip()

# -----------------------------------------------------------------------------
# Find common date range (starting from stocks_df oldest date)
# -----------------------------------------------------------------------------
stocks_min_date = stocks_raw['Date'].min()
stocks_max_date = stocks_raw['Date'].max()
indexes_min_date = indexes_raw['Date'].min()
indexes_max_date = indexes_raw['Date'].max()

print(f"\nStocks date range: {stocks_min_date.date()} to {stocks_max_date.date()}")
print(f"Indexes date range: {indexes_min_date.date()} to {indexes_max_date.date()}")

# Use stocks_df start date as the common starting point
COMMON_START_DATE = stocks_min_date
COMMON_END_DATE = min(stocks_max_date, indexes_max_date)

print(f"\n→ Common analysis period: {COMMON_START_DATE.date()} to {COMMON_END_DATE.date()}")

# -----------------------------------------------------------------------------
# Filter both datasets to common date range
# -----------------------------------------------------------------------------
indexes_df = indexes_raw[
    (indexes_raw['Date'] >= COMMON_START_DATE) &
    (indexes_raw['Date'] <= COMMON_END_DATE)
].copy()

stocks_df = stocks_raw[
    (stocks_raw['Date'] >= COMMON_START_DATE) &
    (stocks_raw['Date'] <= COMMON_END_DATE)
].copy()

print(f"\nFiltered Indexes: {indexes_df.shape}")
print(f"Filtered Stocks: {stocks_df.shape}")

# -----------------------------------------------------------------------------
# Extract benchmark index (NIFTY 500 or NIFTY 50)
# -----------------------------------------------------------------------------
available_indexes = indexes_df['Index'].unique() if 'Index' in indexes_df.columns else []
print(f"\nAvailable indexes: {list(available_indexes)[:10]}...")

BENCHMARK_NAME = None
for idx_name in ['NIFTY 500', 'NIFTY500', 'NIFTY 50', 'NIFTY50']:
    if idx_name in available_indexes:
        BENCHMARK_NAME = idx_name
        break

if BENCHMARK_NAME:
    benchmark_df = indexes_df[indexes_df['Index'] == BENCHMARK_NAME][['Date', 'Close']].copy()
    benchmark_df.rename(columns={'Close': 'Benchmark_Close'}, inplace=True)
    benchmark_df = benchmark_df.sort_values('Date').drop_duplicates('Date')
    print(f"✓ Using {BENCHMARK_NAME} as benchmark ({len(benchmark_df)} days)")
else:
    print("⚠ No standard benchmark found. Will create market-cap weighted proxy.")
    benchmark_df = None

print("\n" + "="*60)
print("DATA ALIGNMENT COMPLETE")
print("="*60)



In [ ]:

# ============================================================================
# CELL 4: CREATE PRICE AND RETURNS MATRICES
# ============================================================================

print("Creating price and returns matrices...\n")

# -----------------------------------------------------------------------------
# Pivot stocks data to wide format
# -----------------------------------------------------------------------------
# Price matrix
price_matrix = stocks_df.pivot_table(
    index='Date',
    columns='Stock',
    values='Close',
    aggfunc='last'
).sort_index()

# Volume matrix
volume_matrix = stocks_df.pivot_table(
    index='Date',
    columns='Stock',
    values='Volume',
    aggfunc='last'
).sort_index()

print(f"Price matrix: {price_matrix.shape}")
print(f"Volume matrix: {volume_matrix.shape}")

# -----------------------------------------------------------------------------
# Data quality filter: require minimum trading days
# -----------------------------------------------------------------------------
MIN_TRADING_DAYS = 252  # At least 1 year of data
MIN_DATA_RATIO = 0.70   # At least 70% data presence

total_days = len(price_matrix)
data_presence = price_matrix.notna().sum()
data_ratio = data_presence / total_days

# Filter stocks with sufficient data
valid_stocks = data_ratio[data_ratio >= MIN_DATA_RATIO].index.tolist()
valid_stocks = [s for s in valid_stocks if data_presence[s] >= MIN_TRADING_DAYS]

print(f"\nStocks with sufficient data (>={MIN_DATA_RATIO*100:.0f}% presence, >={MIN_TRADING_DAYS} days): {len(valid_stocks)}")

# Filter matrices
price_matrix = price_matrix[valid_stocks]
volume_matrix = volume_matrix[valid_stocks]

# -----------------------------------------------------------------------------
# Calculate returns
# -----------------------------------------------------------------------------
returns_matrix = price_matrix.pct_change()

# Forward-fill then back-fill small gaps (max 5 days)
price_matrix = price_matrix.fillna(method='ffill', limit=5).fillna(method='bfill', limit=5)
volume_matrix = volume_matrix.fillna(method='ffill', limit=5).fillna(method='bfill', limit=5)
returns_matrix = price_matrix.pct_change()

# Drop rows with all NaN
returns_matrix = returns_matrix.dropna(how='all')

print(f"\nFinal returns matrix: {returns_matrix.shape}")
print(f"Date range: {returns_matrix.index.min().date()} to {returns_matrix.index.max().date()}")

# Store valid tickers
UNIVERSE = list(returns_matrix.columns)
print(f"\nUniverse size: {len(UNIVERSE)} stocks")

print("\n" + "="*60)
print("MATRICES CREATED")
print("="*60)



In [ ]:

# ============================================================================
# CELL 5: MULTI-FACTOR SCORING SYSTEM
# ============================================================================

print("Calculating Multi-Factor Scores...\n")

import numpy as np

# -----------------------------------------------------------------------------
# FACTOR 1: MOMENTUM (12-month return minus last month)
# Higher is better - captures price trend
# -----------------------------------------------------------------------------
def calculate_momentum(prices, lookback_long=252, lookback_short=21):
    """
    12-1 Momentum: 12-month return excluding the most recent month
    This avoids short-term reversal effects
    """
    ret_12m = prices.pct_change(lookback_long)
    ret_1m = prices.pct_change(lookback_short)
    momentum = ret_12m - ret_1m
    return momentum.iloc[-1]

momentum_scores = calculate_momentum(price_matrix)
print(f"✓ Momentum factor calculated")

# -----------------------------------------------------------------------------
# FACTOR 2: LOW VOLATILITY (Inverse of rolling volatility)
# Lower volatility is better - we'll invert for scoring
# -----------------------------------------------------------------------------
def calculate_volatility(returns, window=252):
    """
    Annualized rolling volatility
    """
    vol = returns.rolling(window=window, min_periods=int(window*0.5)).std() * np.sqrt(252)
    return vol.iloc[-1]

volatility_raw = calculate_volatility(returns_matrix)
# Invert so lower vol = higher score
low_vol_scores = 1 / (volatility_raw + 0.001)  # Add small constant to avoid division by zero
print(f"✓ Low Volatility factor calculated")

# -----------------------------------------------------------------------------
# FACTOR 3: LIQUIDITY (Average Daily Volume * Price)
# Higher liquidity is better for tradability
# -----------------------------------------------------------------------------
def calculate_liquidity(prices, volumes, window=90):
    """
    Average daily value traded (ADV in value terms)
    """
    adv = volumes.rolling(window=window, min_periods=int(window*0.5)).mean()
    avg_price = prices.rolling(window=window, min_periods=int(window*0.5)).mean()
    liquidity = (adv * avg_price).iloc[-1]
    return liquidity

liquidity_scores = calculate_liquidity(price_matrix, volume_matrix)
print(f"✓ Liquidity factor calculated")

# -----------------------------------------------------------------------------
# FACTOR 4: RETURN CONSISTENCY (Sharpe-like ratio)
# Higher consistency is better
# -----------------------------------------------------------------------------
def calculate_consistency(returns, window=126):
    """
    6-month rolling Sharpe ratio (without risk-free rate for simplicity)
    """
    rolling_mean = returns.rolling(window=window, min_periods=int(window*0.5)).mean() * 252
    rolling_std = returns.rolling(window=window, min_periods=int(window*0.5)).std() * np.sqrt(252)
    sharpe = rolling_mean / (rolling_std + 0.001)
    return sharpe.iloc[-1]

consistency_scores = calculate_consistency(returns_matrix)
print(f"✓ Return Consistency factor calculated")

# -----------------------------------------------------------------------------
# FACTOR 5: QUALITY (Price stability - lower drawdown)
# -----------------------------------------------------------------------------
def calculate_quality(prices, window=252):
    """
    Maximum drawdown over the period (inverted - lower drawdown is better)
    """
    rolling_max = prices.rolling(window=window, min_periods=1).max()
    drawdown = (prices - rolling_max) / rolling_max
    max_drawdown = drawdown.rolling(window=window, min_periods=int(window*0.5)).min()
    # Invert: less negative drawdown = higher score
    return -max_drawdown.iloc[-1]

quality_scores = calculate_quality(price_matrix)
print(f"✓ Quality factor calculated")

# -----------------------------------------------------------------------------
# Combine factors into DataFrame
# -----------------------------------------------------------------------------
factor_df = pd.DataFrame({
    'Stock': UNIVERSE,
    'Momentum': momentum_scores.values,
    'LowVol': low_vol_scores.values,
    'Liquidity': liquidity_scores.values,
    'Consistency': consistency_scores.values,
    'Quality': quality_scores.values
}).set_index('Stock')

print(f"\nFactor DataFrame shape: {factor_df.shape}")
print("\nFactor statistics:")
display(factor_df.describe())



In [ ]:

# ============================================================================
# CELL 6: NORMALIZE AND COMBINE FACTORS
# ============================================================================

print("Normalizing and combining factors...\n")

from scipy import stats

# -----------------------------------------------------------------------------
# Z-score normalization (winsorized to handle outliers)
# -----------------------------------------------------------------------------
def winsorize_zscore(series, limits=(0.025, 0.025)):
    """
    Winsorize extreme values then z-score normalize
    """
    # Handle NaN
    valid = series.dropna()
    if len(valid) == 0:
        return series

    # Winsorize at 2.5th and 97.5th percentile
    winsorized = stats.mstats.winsorize(valid, limits=limits)

    # Z-score
    mean = np.mean(winsorized)
    std = np.std(winsorized)
    if std == 0:
        return pd.Series(0, index=series.index)

    z_scores = (series - mean) / std
    return z_scores

# Normalize each factor
factor_normalized = pd.DataFrame(index=factor_df.index)
for col in factor_df.columns:
    factor_normalized[col] = winsorize_zscore(factor_df[col])

print("Normalized factor statistics:")
display(factor_normalized.describe())

# -----------------------------------------------------------------------------
# Factor weights (can be adjusted based on research)
# -----------------------------------------------------------------------------
FACTOR_WEIGHTS = {
    'Momentum': 0.25,      # Trend following
    'LowVol': 0.20,        # Risk reduction
    'Liquidity': 0.15,     # Tradability
    'Consistency': 0.20,   # Stable returns
    'Quality': 0.20        # Drawdown protection
}

print(f"\nFactor weights: {FACTOR_WEIGHTS}")
print(f"Sum of weights: {sum(FACTOR_WEIGHTS.values())}")

# -----------------------------------------------------------------------------
# Calculate composite score
# -----------------------------------------------------------------------------
composite_score = pd.Series(0.0, index=factor_normalized.index)
for factor, weight in FACTOR_WEIGHTS.items():
    composite_score += weight * factor_normalized[factor].fillna(0)

factor_normalized['Composite_Score'] = composite_score

# Rank stocks by composite score
factor_normalized['Rank'] = factor_normalized['Composite_Score'].rank(ascending=False)
factor_normalized = factor_normalized.sort_values('Rank')

print(f"\nTop 20 stocks by composite factor score:")
display(factor_normalized.head(20)[['Composite_Score', 'Rank', 'Momentum', 'LowVol', 'Consistency', 'Quality']])

print("\n" + "="*60)
print("FACTOR SCORING COMPLETE")
print("="*60)




In [ ]:

# ============================================================================
# CELL 7: SELECT TOP STOCKS FOR HRP
# ============================================================================

print("Selecting stocks for HRP optimization...\n")

# -----------------------------------------------------------------------------
# Configuration
# -----------------------------------------------------------------------------
TOP_N_STOCKS = 50  # Number of stocks for portfolio (adjustable)
MIN_LIQUIDITY_PERCENTILE = 20  # Minimum liquidity threshold

# -----------------------------------------------------------------------------
# Apply liquidity filter first
# -----------------------------------------------------------------------------
liquidity_threshold = factor_df['Liquidity'].quantile(MIN_LIQUIDITY_PERCENTILE / 100)
liquid_stocks = factor_df[factor_df['Liquidity'] >= liquidity_threshold].index.tolist()

print(f"Stocks passing liquidity filter: {len(liquid_stocks)}")

# -----------------------------------------------------------------------------
# Select top N by composite score from liquid stocks
# -----------------------------------------------------------------------------
factor_filtered = factor_normalized[factor_normalized.index.isin(liquid_stocks)]
selected_stocks = factor_filtered.head(TOP_N_STOCKS).index.tolist()

print(f"Selected top {len(selected_stocks)} stocks for HRP")

# -----------------------------------------------------------------------------
# Prepare returns for selected stocks
# -----------------------------------------------------------------------------
returns_selected = returns_matrix[selected_stocks].dropna()

print(f"\nReturns matrix for HRP: {returns_selected.shape}")
print(f"\nSelected stocks:")
for i, stock in enumerate(selected_stocks, 1):
    score = factor_normalized.loc[stock, 'Composite_Score']
    print(f"  {i:2}. {stock}: {score:.4f}")

print("\n" + "="*60)
print("STOCK SELECTION COMPLETE")
print("="*60)


In [ ]:

# ============================================================================
# CELL 8: HIERARCHICAL RISK PARITY (HRP) IMPLEMENTATION
# ============================================================================

print("Implementing Hierarchical Risk Parity (HRP)...\n")

from scipy.cluster.hierarchy import linkage, dendrogram, leaves_list
from scipy.spatial.distance import squareform
import matplotlib.pyplot as plt

# -----------------------------------------------------------------------------
# HRP Helper Functions
# -----------------------------------------------------------------------------

def get_correlation_distance(corr_matrix):
    """
    Convert correlation matrix to distance matrix
    Distance = sqrt(0.5 * (1 - correlation))
    """
    distance = np.sqrt(0.5 * (1 - corr_matrix))
    return distance


def get_quasi_diagonal(link):
    """
    Sort clustered items by distance (quasi-diagonalization)
    """
    return leaves_list(link)


def get_cluster_variance(cov, cluster_items):
    """
    Calculate variance of an inverse-variance weighted portfolio
    for a cluster of assets
    """
    cov_slice = cov[np.ix_(cluster_items, cluster_items)]

    # Inverse variance weights within cluster
    ivp_weights = 1 / np.diag(cov_slice)
    ivp_weights = ivp_weights / ivp_weights.sum()

    # Cluster variance
    cluster_var = np.dot(ivp_weights, np.dot(cov_slice, ivp_weights))
    return cluster_var


def recursive_bisection(cov, sorted_idx):
    """
    Recursive bisection to allocate weights based on inverse variance
    """
    n = len(sorted_idx)
    weights = np.ones(n)

    # List of clusters to process: (start, end)
    clusters = [(0, n)]

    while clusters:
        start, end = clusters.pop(0)

        if end - start <= 1:
            continue

        # Split cluster in half
        mid = (start + end) // 2

        # Left and right cluster items (indices in original order)
        left_items = sorted_idx[start:mid]
        right_items = sorted_idx[mid:end]

        # Calculate cluster variances
        left_var = get_cluster_variance(cov, left_items)
        right_var = get_cluster_variance(cov, right_items)

        # Allocate inversely proportional to variance
        total_var = left_var + right_var
        if total_var > 0:
            alpha = 1 - left_var / total_var  # Weight for left cluster
        else:
            alpha = 0.5

        # Apply allocation
        weights[left_items] *= alpha
        weights[right_items] *= (1 - alpha)

        # Add sub-clusters to process
        clusters.append((start, mid))
        clusters.append((mid, end))

    return weights


def hrp_portfolio(returns):
    """
    Main HRP function

    Steps:
    1. Calculate correlation and covariance matrices
    2. Build hierarchical clustering tree
    3. Quasi-diagonalize the covariance matrix
    4. Recursive bisection to get weights
    """
    # Step 1: Correlation and covariance
    corr = returns.corr().values
    cov = returns.cov().values * 252  # Annualize

    # Handle any NaN/Inf in correlation
    corr = np.nan_to_num(corr, nan=0, posinf=1, neginf=-1)
    np.fill_diagonal(corr, 1)

    # Step 2: Distance matrix and hierarchical clustering
    distance = get_correlation_distance(corr)

    # Convert to condensed form for linkage
    np.fill_diagonal(distance, 0)  # Diagonal should be 0
    condensed_dist = squareform(distance)

    # Hierarchical clustering (Ward's method)
    link = linkage(condensed_dist, method='ward')

    # Step 3: Quasi-diagonalization
    sorted_idx = get_quasi_diagonal(link)

    # Step 4: Recursive bisection
    weights = recursive_bisection(cov, sorted_idx)

    # Normalize weights
    weights = weights / weights.sum()

    return weights, link, sorted_idx


# -----------------------------------------------------------------------------
# Run HRP
# -----------------------------------------------------------------------------
print("Running HRP optimization...")

hrp_weights, linkage_matrix, sorted_indices = hrp_portfolio(returns_selected)

# Create portfolio DataFrame
hrp_portfolio_df = pd.DataFrame({
    'Stock': selected_stocks,
    'HRP_Weight': hrp_weights
}).sort_values('HRP_Weight', ascending=False)

print(f"\n✓ HRP optimization complete")
print(f"\nHRP Portfolio weights (Top 20):")
display(hrp_portfolio_df.head(20).style.format({'HRP_Weight': '{:.4f}'}))

# -----------------------------------------------------------------------------
# Visualize dendrogram
# -----------------------------------------------------------------------------
plt.figure(figsize=(14, 6))
dendrogram(
    linkage_matrix,
    labels=selected_stocks,
    leaf_rotation=90,
    leaf_font_size=8
)
plt.title('Hierarchical Clustering Dendrogram (HRP)')
plt.xlabel('Stocks')
plt.ylabel('Distance')
plt.tight_layout()
plt.show()

print("\n" + "="*60)
print("HRP OPTIMIZATION COMPLETE")
print("="*60)




In [ ]:
# ============================================================================
# CELL 9: APPLY RISK CONTROLS AND CONSTRAINTS
# ============================================================================

print("Applying risk controls and constraints...\n")

import yfinance as yf

# -----------------------------------------------------------------------------
# Risk Control Parameters
# -----------------------------------------------------------------------------
MAX_SINGLE_STOCK = 0.08      # 8% max per stock
MIN_SINGLE_STOCK = 0.005    # 0.5% min per stock (avoid dust positions)
SECTOR_CAP = 0.25           # 25% max per sector
MAX_POSITIONS = 40          # Maximum number of positions

print(f"Risk Control Parameters:")
print(f"  - Max single stock: {MAX_SINGLE_STOCK*100:.1f}%")
print(f"  - Min single stock: {MIN_SINGLE_STOCK*100:.1f}%")
print(f"  - Sector cap: {SECTOR_CAP*100:.1f}%")
print(f"  - Max positions: {MAX_POSITIONS}")

# -----------------------------------------------------------------------------
# Fetch sector information
# -----------------------------------------------------------------------------
print("\nFetching sector information...")

def get_sector_map(tickers):
    """
    Fetch sector information using yfinance
    """
    sector_map = {}

    for ticker in tickers:
        try:
            # Try with .NS suffix for NSE stocks
            yf_ticker = f"{ticker}.NS" if "." not in ticker else ticker
            info = yf.Ticker(yf_ticker).info
            sector = info.get('sector') or info.get('industry') or 'Others'
            sector_map[ticker] = sector
        except:
            sector_map[ticker] = 'Others'

    return sector_map

# Get sectors for selected stocks
sector_map = get_sector_map(hrp_portfolio_df['Stock'].tolist())
hrp_portfolio_df['Sector'] = hrp_portfolio_df['Stock'].map(sector_map)

print(f"Sector breakdown:")
sector_counts = hrp_portfolio_df.groupby('Sector')['Stock'].count()
display(sector_counts)

# -----------------------------------------------------------------------------
# Apply weight constraints
# -----------------------------------------------------------------------------
print("\nApplying weight constraints...")

def apply_constraints(portfolio_df, max_weight, min_weight, sector_cap):
    """
    Apply position and sector constraints
    """
    df = portfolio_df.copy()

    # Step 1: Cap individual weights
    df['Constrained_Weight'] = df['HRP_Weight'].clip(upper=max_weight)

    # Step 2: Apply sector caps
    for sector in df['Sector'].unique():
        sector_mask = df['Sector'] == sector
        sector_weight = df.loc[sector_mask, 'Constrained_Weight'].sum()

        if sector_weight > sector_cap:
            # Scale down proportionally
            scale = sector_cap / sector_weight
            df.loc[sector_mask, 'Constrained_Weight'] *= scale

    # Step 3: Remove positions below minimum
    df = df[df['Constrained_Weight'] >= min_weight].copy()

    # Step 4: Renormalize to sum to 1
    total = df['Constrained_Weight'].sum()
    if total > 0:
        df['Final_Weight'] = df['Constrained_Weight'] / total
    else:
        df['Final_Weight'] = df['Constrained_Weight']

    return df.sort_values('Final_Weight', ascending=False)

# Apply constraints
final_portfolio = apply_constraints(
    hrp_portfolio_df,
    MAX_SINGLE_STOCK,
    MIN_SINGLE_STOCK,
    SECTOR_CAP
)

# Limit to max positions
if len(final_portfolio) > MAX_POSITIONS:
    final_portfolio = final_portfolio.head(MAX_POSITIONS)
    # Renormalize
    final_portfolio['Final_Weight'] = final_portfolio['Final_Weight'] / final_portfolio['Final_Weight'].sum()

print(f"\n✓ Constraints applied")
print(f"Final portfolio: {len(final_portfolio)} positions")

# -----------------------------------------------------------------------------
# Display final portfolio
# -----------------------------------------------------------------------------
print(f"\n{'='*60}")
print("FINAL MULTI-FACTOR + HRP PORTFOLIO")
print(f"{'='*60}\n")

display(final_portfolio[['Stock', 'Sector', 'Final_Weight']].style.format({'Final_Weight': '{:.4f}'}))

# Sector allocation
print("\nSector Allocation:")
sector_allocation = final_portfolio.groupby('Sector')['Final_Weight'].sum().sort_values(ascending=False)
display(sector_allocation.to_frame('Weight').style.format({'Weight': '{:.4f}'}))



In [ ]:

# ============================================================================
# CELL 10: PORTFOLIO METRICS AND BACKTESTING
# ============================================================================

print("Calculating portfolio metrics and backtesting...\n")

import matplotlib.pyplot as plt
import seaborn as sns

# -----------------------------------------------------------------------------
# Get final weights and returns
# -----------------------------------------------------------------------------
final_tickers = final_portfolio['Stock'].tolist()
final_weights = final_portfolio.set_index('Stock')['Final_Weight']

# Get returns for final portfolio stocks
portfolio_returns = returns_matrix[final_tickers].copy()
portfolio_returns = portfolio_returns.dropna()

# Calculate weighted portfolio returns
weights_array = final_weights.reindex(final_tickers).values
portfolio_daily_returns = portfolio_returns.dot(weights_array)

# -----------------------------------------------------------------------------
# Calculate benchmark returns if available
# -----------------------------------------------------------------------------
if benchmark_df is not None:
    benchmark_prices = benchmark_df.set_index('Date')['Benchmark_Close']
    benchmark_returns = benchmark_prices.pct_change().dropna()

    # Align dates
    common_dates = portfolio_daily_returns.index.intersection(benchmark_returns.index)
    portfolio_daily_returns = portfolio_daily_returns.loc[common_dates]
    benchmark_returns = benchmark_returns.loc[common_dates]
else:
    # Create equal-weight benchmark from universe
    benchmark_returns = returns_matrix[UNIVERSE].mean(axis=1)
    common_dates = portfolio_daily_returns.index.intersection(benchmark_returns.index)
    portfolio_daily_returns = portfolio_daily_returns.loc[common_dates]
    benchmark_returns = benchmark_returns.loc[common_dates]

# -----------------------------------------------------------------------------
# Calculate performance metrics
# -----------------------------------------------------------------------------
RISK_FREE_RATE = 0.06  # 6% annual risk-free rate for India
TRADING_DAYS = 252

def calculate_metrics(returns, rf_rate=0.06):
    """
    Calculate key portfolio metrics
    """
    # Annualized return
    total_return = (1 + returns).prod() - 1
    n_years = len(returns) / TRADING_DAYS
    ann_return = (1 + total_return) ** (1 / n_years) - 1 if n_years > 0 else 0

    # Annualized volatility
    ann_vol = returns.std() * np.sqrt(TRADING_DAYS)

    # Sharpe ratio
    sharpe = (ann_return - rf_rate) / ann_vol if ann_vol > 0 else 0

    # Sortino ratio (downside deviation)
    downside_returns = returns[returns < 0]
    downside_std = downside_returns.std() * np.sqrt(TRADING_DAYS) if len(downside_returns) > 0 else ann_vol
    sortino = (ann_return - rf_rate) / downside_std if downside_std > 0 else 0

    # Maximum drawdown
    cumulative = (1 + returns).cumprod()
    rolling_max = cumulative.expanding().max()
    drawdown = (cumulative - rolling_max) / rolling_max
    max_drawdown = drawdown.min()

    # Calmar ratio
    calmar = ann_return / abs(max_drawdown) if max_drawdown != 0 else 0

    return {
        'Total Return': total_return,
        'Ann. Return': ann_return,
        'Ann. Volatility': ann_vol,
        'Sharpe Ratio': sharpe,
        'Sortino Ratio': sortino,
        'Max Drawdown': max_drawdown,
        'Calmar Ratio': calmar
    }

# Calculate metrics for portfolio and benchmark
portfolio_metrics = calculate_metrics(portfolio_daily_returns, RISK_FREE_RATE)
benchmark_metrics = calculate_metrics(benchmark_returns, RISK_FREE_RATE)

# -----------------------------------------------------------------------------
# Display metrics comparison
# -----------------------------------------------------------------------------
print(f"{'='*60}")
print("PERFORMANCE METRICS COMPARISON")
print(f"{'='*60}\n")

metrics_comparison = pd.DataFrame({
    'Portfolio': portfolio_metrics,
    'Benchmark': benchmark_metrics
}).T

# Format for display
display(metrics_comparison.style.format({
    'Total Return': '{:.2%}',
    'Ann. Return': '{:.2%}',
    'Ann. Volatility': '{:.2%}',
    'Sharpe Ratio': '{:.2f}',
    'Sortino Ratio': '{:.2f}',
    'Max Drawdown': '{:.2%}',
    'Calmar Ratio': '{:.2f}'
}))

# Alpha and Beta
cov_with_benchmark = np.cov(portfolio_daily_returns.values, benchmark_returns.values)
beta = cov_with_benchmark[0, 1] / cov_with_benchmark[1, 1] if cov_with_benchmark[1, 1] > 0 else 1
alpha = portfolio_metrics['Ann. Return'] - (RISK_FREE_RATE + beta * (benchmark_metrics['Ann. Return'] - RISK_FREE_RATE))

print(f"\nPortfolio Beta: {beta:.4f}")
print(f"Portfolio Alpha: {alpha:.4f} ({alpha*100:.2f}%)")



In [ ]:

# ============================================================================
# CELL 11: VISUALIZATION
# ============================================================================

print("Generating visualizations...\n")

import matplotlib.pyplot as plt
import seaborn as sns

# Set style
plt.style.use('seaborn-v0_8-whitegrid')
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# -----------------------------------------------------------------------------
# Plot 1: Cumulative Returns
# -----------------------------------------------------------------------------
ax1 = axes[0, 0]
portfolio_cumulative = (1 + portfolio_daily_returns).cumprod()
benchmark_cumulative = (1 + benchmark_returns).cumprod()

ax1.plot(portfolio_cumulative.index, portfolio_cumulative.values, label='Multi-Factor HRP Portfolio', linewidth=2)
ax1.plot(benchmark_cumulative.index, benchmark_cumulative.values, label='Benchmark', linewidth=2, alpha=0.7)
ax1.set_title('Cumulative Returns', fontsize=12, fontweight='bold')
ax1.set_xlabel('Date')
ax1.set_ylabel('Growth of ₹1')
ax1.legend()
ax1.grid(True, alpha=0.3)

# -----------------------------------------------------------------------------
# Plot 2: Drawdown
# -----------------------------------------------------------------------------
ax2 = axes[0, 1]
rolling_max = portfolio_cumulative.expanding().max()
drawdown = (portfolio_cumulative - rolling_max) / rolling_max

ax2.fill_between(drawdown.index, drawdown.values, 0, alpha=0.3, color='red')
ax2.plot(drawdown.index, drawdown.values, color='red', linewidth=1)
ax2.set_title('Portfolio Drawdown', fontsize=12, fontweight='bold')
ax2.set_xlabel('Date')
ax2.set_ylabel('Drawdown')
ax2.grid(True, alpha=0.3)

# -----------------------------------------------------------------------------
# Plot 3: Portfolio Weights
# -----------------------------------------------------------------------------
ax3 = axes[1, 0]
top_n = min(15, len(final_portfolio))
top_holdings = final_portfolio.head(top_n)

colors = plt.cm.viridis(np.linspace(0, 0.8, top_n))
bars = ax3.barh(range(top_n), top_holdings['Final_Weight'].values, color=colors)
ax3.set_yticks(range(top_n))
ax3.set_yticklabels(top_holdings['Stock'].values)
ax3.set_xlabel('Weight')
ax3.set_title(f'Top {top_n} Holdings', fontsize=12, fontweight='bold')
ax3.invert_yaxis()

# Add percentage labels
for i, (bar, weight) in enumerate(zip(bars, top_holdings['Final_Weight'].values)):
    ax3.text(weight + 0.002, i, f'{weight:.1%}', va='center', fontsize=9)

# -----------------------------------------------------------------------------
# Plot 4: Sector Allocation
# -----------------------------------------------------------------------------
ax4 = axes[1, 1]
sector_weights = final_portfolio.groupby('Sector')['Final_Weight'].sum().sort_values(ascending=False)

colors = plt.cm.Set3(np.linspace(0, 1, len(sector_weights)))
wedges, texts, autotexts = ax4.pie(
    sector_weights.values,
    labels=sector_weights.index,
    autopct='%1.1f%%',
    colors=colors,
    startangle=90
)
ax4.set_title('Sector Allocation', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.savefig('portfolio_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n✓ Saved visualization to 'portfolio_analysis.png'")



In [ ]:

# ============================================================================
# CELL 12: ROLLING PERFORMANCE ANALYSIS
# ============================================================================

print("Calculating rolling performance metrics...\n")

# Rolling window size
ROLLING_WINDOW = 252  # 1 year

# Calculate rolling metrics
rolling_returns = portfolio_daily_returns.rolling(window=ROLLING_WINDOW).mean() * TRADING_DAYS
rolling_vol = portfolio_daily_returns.rolling(window=ROLLING_WINDOW).std() * np.sqrt(TRADING_DAYS)
rolling_sharpe = (rolling_returns - RISK_FREE_RATE) / rolling_vol

# Same for benchmark
bench_rolling_returns = benchmark_returns.rolling(window=ROLLING_WINDOW).mean() * TRADING_DAYS
bench_rolling_vol = benchmark_returns.rolling(window=ROLLING_WINDOW).std() * np.sqrt(TRADING_DAYS)
bench_rolling_sharpe = (bench_rolling_returns - RISK_FREE_RATE) / bench_rolling_vol

# Plot rolling metrics
fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

# Rolling Returns
axes[0].plot(rolling_returns.index, rolling_returns.values, label='Portfolio', linewidth=2)
axes[0].plot(bench_rolling_returns.index, bench_rolling_returns.values, label='Benchmark', linewidth=2, alpha=0.7)
axes[0].axhline(y=0, color='gray', linestyle='--', alpha=0.5)
axes[0].set_title('Rolling 1-Year Annualized Return', fontweight='bold')
axes[0].set_ylabel('Return')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Rolling Volatility
axes[1].plot(rolling_vol.index, rolling_vol.values, label='Portfolio', linewidth=2)
axes[1].plot(bench_rolling_vol.index, bench_rolling_vol.values, label='Benchmark', linewidth=2, alpha=0.7)
axes[1].set_title('Rolling 1-Year Volatility', fontweight='bold')
axes[1].set_ylabel('Volatility')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Rolling Sharpe
axes[2].plot(rolling_sharpe.index, rolling_sharpe.values, label='Portfolio', linewidth=2)
axes[2].plot(bench_rolling_sharpe.index, bench_rolling_sharpe.values, label='Benchmark', linewidth=2, alpha=0.7)
axes[2].axhline(y=0, color='gray', linestyle='--', alpha=0.5)
axes[2].axhline(y=1, color='green', linestyle='--', alpha=0.5, label='Sharpe = 1')
axes[2].set_title('Rolling 1-Year Sharpe Ratio', fontweight='bold')
axes[2].set_ylabel('Sharpe Ratio')
axes[2].set_xlabel('Date')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('rolling_performance.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n✓ Saved rolling performance to 'rolling_performance.png'")



In [ ]:

# ============================================================================
# CELL 13: EXPORT FINAL PORTFOLIO
# ============================================================================

print("Exporting final portfolio...\n")

# -----------------------------------------------------------------------------
# Add factor scores to final portfolio
# -----------------------------------------------------------------------------
export_portfolio = final_portfolio.copy()

# Add factor scores
for col in ['Momentum', 'LowVol', 'Consistency', 'Quality', 'Composite_Score']:
    if col in factor_normalized.columns:
        export_portfolio[col] = export_portfolio['Stock'].map(
            factor_normalized[col].to_dict()
        )

# Reorder columns
column_order = ['Stock', 'Sector', 'Final_Weight', 'HRP_Weight',
                'Composite_Score', 'Momentum', 'LowVol', 'Consistency', 'Quality']
column_order = [c for c in column_order if c in export_portfolio.columns]
export_portfolio = export_portfolio[column_order]

# -----------------------------------------------------------------------------
# Save to CSV
# -----------------------------------------------------------------------------
export_portfolio.to_csv('multi_factor_hrp_portfolio.csv', index=False)
print("✓ Saved portfolio to 'multi_factor_hrp_portfolio.csv'")

# -----------------------------------------------------------------------------
# Create summary report
# -----------------------------------------------------------------------------
summary = {
    'Strategy': 'Multi-Factor + HRP',
    'Analysis Period': f"{portfolio_daily_returns.index.min().date()} to {portfolio_daily_returns.index.max().date()}",
    'Number of Positions': len(final_portfolio),
    'Total Return': f"{portfolio_metrics['Total Return']:.2%}",
    'Annualized Return': f"{portfolio_metrics['Ann. Return']:.2%}",
    'Annualized Volatility': f"{portfolio_metrics['Ann. Volatility']:.2%}",
    'Sharpe Ratio': f"{portfolio_metrics['Sharpe Ratio']:.2f}",
    'Sortino Ratio': f"{portfolio_metrics['Sortino Ratio']:.2f}",
    'Max Drawdown': f"{portfolio_metrics['Max Drawdown']:.2%}",
    'Beta': f"{beta:.4f}",
    'Alpha': f"{alpha:.4f}",
    'Top Holding': f"{final_portfolio.iloc[0]['Stock']} ({final_portfolio.iloc[0]['Final_Weight']:.2%})",
    'Top Sector': f"{sector_allocation.index[0]} ({sector_allocation.iloc[0]:.2%})"
}

summary_df = pd.DataFrame(list(summary.items()), columns=['Metric', 'Value'])
summary_df.to_csv('portfolio_summary.csv', index=False)
print("✓ Saved summary to 'portfolio_summary.csv'")

# -----------------------------------------------------------------------------
# Display final summary
# -----------------------------------------------------------------------------
print(f"\n{'='*60}")
print("MULTI-FACTOR + HRP PORTFOLIO SUMMARY")
print(f"{'='*60}\n")

for metric, value in summary.items():
    print(f"{metric:.<30} {value}")

print(f"\n{'='*60}")
print("FINAL PORTFOLIO HOLDINGS")
print(f"{'='*60}\n")

display(export_portfolio.style.format({
    'Final_Weight': '{:.4f}',
    'HRP_Weight': '{:.4f}',
    'Composite_Score': '{:.4f}',
    'Momentum': '{:.4f}',
    'LowVol': '{:.4f}',
    'Consistency': '{:.4f}',
    'Quality': '{:.4f}'
}))

print("\n" + "="*60)
print("STRATEGY IMPLEMENTATION COMPLETE")
print("="*60)



In [ ]:

# ============================================================================
# CELL 14: DOWNLOAD FILES (Colab only)
# ============================================================================

try:
    from google.colab import files

    print("Preparing files for download...\n")

    # Download portfolio files
    files.download('multi_factor_hrp_portfolio.csv')
    files.download('portfolio_summary.csv')
    files.download('portfolio_analysis.png')
    files.download('rolling_performance.png')

    print("\n✓ All files downloaded successfully!")

except ImportError:
    print("Running locally - files saved to current directory.")
    print("\nFiles created:")
    print("  - multi_factor_hrp_portfolio.csv")
    print("  - portfolio_summary.csv")
    print("  - portfolio_analysis.png")
    print("  - rolling_performance.png")
